# NYC data import

## 1. Create Schema and Volume where the data will live

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.nyc_taxi_raw")
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.nyc_taxi_raw.landing")

## 2. Load raw data
Get data from the cloud and store into the volume created in last step

In [0]:
import urllib.request
base = "https://d37ci6vzurychx.cloudfront.net"
vol = "/Volumes/workspace/nyc_taxi_raw/landing"

for color in ["yellow", "green"]:
    for m in ["2024-01", "2024-02", "2024-03"]:
        f = f"{color}_tripdata_{m}.parquet"
        urllib.request.urlretrieve(f"{base}/trip-data/{f}", f"{vol}/{f}")

urllib.request.urlretrieve(f"{base}/misc/taxi_zone_lookup.csv", f"{vol}/taxi_zone_lookup.csv")

## 3.1 Display the data loaded into volume

In [0]:
vol = "/Volumes/workspace/nyc_taxi_raw/landing"

# Check the files arrived and their sizes
display(dbutils.fs.ls(vol))

## 3.2 Display the yellow trip data 
review the trip file structure and sample data by using display and limit

In [0]:
# Yellow trips: schema, row count, first rows
yellow = spark.read.parquet(f"{vol}/yellow_tripdata_*.parquet")
yellow.printSchema()
print(f"Yellow rows: {yellow.count():,}")
display(yellow.limit(20))

## 3.3 Display the green trip data
Similar to the cell above, apply to green trip data

In [0]:
# Green trips
green = spark.read.parquet(f"{vol}/green_tripdata_*.parquet")
green.printSchema()
print(f"Green rows: {green.count():,}")
display(green.limit(20))

## 3.4 Review the zone data

In [0]:
# Zone lookup (CSV)
zones = spark.read.option("header", True).csv(f"{vol}/taxi_zone_lookup.csv")
display(zones)

## 4.1 Convert raw data into tables

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.nyc_taxi_raw.yellow_tripdata AS
SELECT *, _metadata.file_name AS _source_file
FROM read_files('/Volumes/workspace/nyc_taxi_raw/landing/yellow_tripdata_*.parquet');

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.nyc_taxi_raw.green_tripdata AS
SELECT *, _metadata.file_name AS _source_file
FROM read_files('/Volumes/workspace/nyc_taxi_raw/landing/green_tripdata_*.parquet');

In [0]:
%sql
-- repeat for green_tripdata
CREATE OR REPLACE TABLE workspace.nyc_taxi_raw.taxi_zone_lookup AS
SELECT * FROM read_files('/Volumes/workspace/nyc_taxi_raw/landing/taxi_zone_lookup.csv', format => 'csv', header => true);

## 4.2 check the raw tables

In [0]:
%sql
-- List the tables in the schema
SHOW TABLES IN workspace.nyc_taxi_raw;

In [0]:
%sql
-- Row counts per table, compared with the files
SELECT 'yellow_tripdata' AS table_name, COUNT(*) AS row_count FROM workspace.nyc_taxi_raw.yellow_tripdata
UNION ALL
SELECT 'green_tripdata', COUNT(*) FROM workspace.nyc_taxi_raw.green_tripdata
UNION ALL
SELECT 'taxi_zone_lookup', COUNT(*) FROM workspace.nyc_taxi_raw.taxi_zone_lookup;

In [0]:
%sql
-- Preview each table
SELECT * FROM workspace.nyc_taxi_raw.yellow_tripdata LIMIT 20;

In [0]:
%sql
SELECT * FROM workspace.nyc_taxi_raw.green_tripdata LIMIT 20;

In [0]:
%sql
SELECT * FROM workspace.nyc_taxi_raw.taxi_zone_lookup;

In [0]:
%sql
-- Rows per source file: each month should be present
SELECT _source_file, COUNT(*) AS row_count
FROM workspace.nyc_taxi_raw.yellow_tripdata
GROUP BY _source_file
ORDER BY _source_file;

In [0]:
%sql
-- Check column types
DESCRIBE TABLE workspace.nyc_taxi_raw.yellow_tripdata;